In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
def clean_generic_name(name):
    if pd.isna(name) or name == 'nan':
        return name
    
    # 1. 統一標點符號：全型斜線與逗號轉半型，移除常見括號內容
    name = name.replace('／', '/').replace('，', ',')
    name = re.sub(r'[\(\uff08].*?[\)\uff09]', '', name)
    
    # 2. 強力移除劑量與單位 (處理 100,000, 1%, 80 mg/ml, 2.4 MIU 等)
    # 匹配數字、逗號、點、百分比，後接常見容量單位
    name = re.sub(r'[\d\.,\s]+(g|mg|ml|units|miu|％|%)\b', ' ', name, flags=re.IGNORECASE)
    # 針對單獨殘留的 /ml 或 %
    name = re.sub(r'/\s*ml|%', '', name, flags=re.IGNORECASE)
    
    # 3. 移除特定劑型關鍵字與無義詞
    forms = ['inj', 'tab', 'cap', 'susp', 'iv', 'oral', 'for', 'solution', 'hydrate', 'sodium', 'benzathine']
    pattern_forms = r'\b(' + '|'.join(forms) + r')\b'
    name = re.sub(pattern_forms, '', name, flags=re.IGNORECASE)

    # 4. 特定藥名結構處理 (依據您的要求)
    # Amphotericin B liposome -> Amphotericin B/liposome
    name = re.sub(r'Amphotericin B liposome', 'Amphotericin B/liposome', name, flags=re.IGNORECASE)

    # 5. 【核心強化】消除斜線 (/) 前後的任何空格
    # \s* 代表 0 到多個空白字元
    name = re.sub(r'\s*/\s*', '/', name)

    
    # 5. 清理殘留符號：移除多餘空格、末尾點號與斜線
    name = re.sub(r'\s+', ' ', name) # 多空格轉單空格
    name = name.strip(' ./,')       # 移除前後的空格、點、斜線、逗號
    
    # 6. 字典對照 (處理特殊轉換)
    mapping = {
        'R +I': 'ifampin/Isoniazid',
        'R 300 +I 150': 'ifampin/Isoniazid',
        'Penicillin G .': 'Penicillin G',
        'Penicillin G benzathine': 'Penicillin G',
        'Baktar': 'Sulfamethoxazole/Trimethoprim',
        'Clindamycin 1 ％' : 'Clindamycin',
        'Penicillin 5 MU' : 'Penicillin',
        'Rifampin, Isoniazid and Ethambutol' : 'Rifampin/Isoniazid/Ethambutol',
        'Minocycline injection' : 'Minocycline',
        'Amoxicillin/Clavulanate': 'Amoxicillin/Clavulanic acid'
    }
    
    # 如果完全符合字典 key，或是處理後變成 key 的樣子就轉換
    return mapping.get(name, name)

In [3]:
df2024 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2024\2024Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})
df2025 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2025\2025Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})

df2024['VERIFYDATE'] = pd.to_datetime(df2024['VERIFYDATE'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['VERIFYDATE'] = pd.to_datetime(df2025['VERIFYDATE'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['STARTTIME'] = pd.to_datetime(df2024['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['ENDTIME'] = pd.to_datetime(df2024['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['STARTTIME'] = pd.to_datetime(df2025['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['ENDTIME'] = pd.to_datetime(df2025['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')

table14 = pd.concat([df2024, df2025], ignore_index=True)

In [4]:
table14 = table14.dropna(how='all')
table14 = table14[table14['GENERICNAME'].notna()]
table14 = table14.loc[:, ~table14.columns.str.contains('^Unnamed')]

In [5]:
len(table14), len(table14['ACCOUNTNO'].unique())

(86059, 27968)

In [6]:
table14['STARTTIME'] = pd.to_datetime(table14['STARTTIME'], format='%Y%m%d%H%M', errors='coerce')
table14['ENDTIME'] = pd.to_datetime(table14['ENDTIME'], format='%Y%m%d%H%M', errors='coerce')

In [7]:
# table14[table14['ACCOUNTNO'] == 'I11300000002']

In [8]:
table14['GENERICNAME_Clear'] = table14['GENERICNAME'].apply(clean_generic_name)

In [9]:
table14['GENERICNAME_Clear'].unique()

array(['Flomoxef', 'Tenofovir alafenamide', 'Cefixime',
       'Amoxicillin/Clavulanic acid', 'Metronidazole', 'Cefazolin',
       'Ceftriaxone', 'Cefoperazone/sulbactam', 'Gentamicin',
       'Cefadroxil', 'Oseltamivir', 'Baloxavir marboxil', 'Clindamycin',
       'Piperacillin/Tazobactam', 'Cefepime', 'Azithromycin',
       'Levofloxacin', 'Acyclovir', 'Cefuroxime', 'Ciprofloxacin',
       'Doxycycline', 'Peramivir', 'Vancomycin', 'Amoxicillin',
       'Valaciclovir', 'Nystatin', 'Ceftazidime', 'Penicillin',
       'Doripenem', 'Ertapenem', 'Cefotaxime', 'Cephalexin', 'Oxacillin',
       'Sulfamethoxazole/Trimethoprim', 'Ampicillin', 'Itraconazole',
       'Fluconazole', 'Amikacin', 'Moxifloxacin', 'Clarithromycin',
       'Anidulafungin', 'Ceftazidime/Avibactam', 'Fenticonazole',
       'Fosfomycin', 'Pipemidic acid', 'Cefoxitin',
       'Rifampin/Isoniazid/Ethambutol', 'Pyrazinamide', 'Teicoplanin',
       'Meropenem', 'Minocycline', 'ifampin/Isoniazid', 'Ceftizoxime',
       'Famc

In [10]:
table14['GENERICNAME_Clear'][table14['GENERICNAME_Clear'].isna() == True]

Series([], Name: GENERICNAME_Clear, dtype: object)

In [11]:
table14[['ACCOUNTNO', 'GENERICNAME_Clear']]

,ACCOUNTNO,GENERICNAME_Clear
0,I11300000002,Flomoxef
1,I11300000002,Flomoxef
2,I11300000002,Flomoxef
3,I11300000002,Tenofovir alafenamide
4,I11300000002,Tenofovir alafenamide
...,...,...
86055,I11400060720,Piperacillin/Tazobactam
86056,I11400060731,Flomoxef
86057,I11400060731,Flomoxef
86058,I11400060731,Flomoxef


In [12]:
infection_cols = [
    "INFECTIONSITE1",
    "INFECTIONSITE2",
    "INFECTIONSITE3",
    "INFECTIONSITE4",
    "INFECTIONSITE5",
    "INFECTIONSITE9"
]

for col in infection_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [13]:
table14['INFECTIONSITE1']

0        1
1        0
2        1
3        0
4        0
        ..
86055    1
86056    1
86057    0
86058    1
86059    1
Name: INFECTIONSITE1, Length: 86059, dtype: int32

In [14]:
BACTERIA_cols = [
    "BACTERIA1",
    "BACTERIA2",
    "BACTERIA3",
    "BACTERIA4",
    "BACTERIA5",
    "BACTERIA9"
]

for col in BACTERIA_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [15]:
table14['BACTERIA1']

0        0
1        0
2        1
3        0
4        0
        ..
86055    0
86056    0
86057    0
86058    0
86059    0
Name: BACTERIA1, Length: 86059, dtype: int32

In [16]:
table14['AUTIBIOTICRANK'].unique()

array(['A32', 'A11', 'A21', 'A42', 'A22', 'A52', 'C42'], dtype=object)

In [17]:
rank_mapping = {
    'A11': 1,
    'A21': 2,
    'A22': 2,
    'A32': 2,
    'A42': 3,
    'A52': 3,
    'C42': 3
}

In [18]:
table14['AUTIBIOTIC_GROUP'] = table14['AUTIBIOTICRANK'].map(rank_mapping)

In [19]:
# table14 = table14.sort_values(["ACCOUNTNO", "STARTTIME"])
# table14_last = table14.groupby("ACCOUNTNO").tail(1)
# table14_last = table14_last.reset_index(drop=True)

In [20]:
# len(table14_last['GENERICNAME_Clear'].unique())

In [21]:
table14_last = table14[['ACCOUNTNO', 'GENERICNAME_Clear', 'AUTIBIOTIC_GROUP']].drop_duplicates()

In [22]:
table14_last

,ACCOUNTNO,GENERICNAME_Clear,AUTIBIOTIC_GROUP
0,I11300000002,Flomoxef,2
3,I11300000002,Tenofovir alafenamide,1
5,I11300000002,Cefixime,2
7,I11300000003,Amoxicillin/Clavulanic acid,2
8,I11300000008,Cefixime,2
...,...,...,...
86048,I11400060701,Cefuroxime,2
86049,I11400060701,Flomoxef,2
86050,I11400060720,Piperacillin/Tazobactam,2
86054,I11400060720,Azithromycin,2


In [23]:
abx14 = (table14_last.assign(value=1)
                     .pivot_table(index=['ACCOUNTNO'],columns='GENERICNAME_Clear',values='value',fill_value=0))

In [24]:
add = table14_last.groupby('ACCOUNTNO')['AUTIBIOTIC_GROUP'].max().reset_index()
abx14 = abx14.merge(add, on='ACCOUNTNO', how='left')

In [25]:
# infection_cols = ['INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3', 
#                   'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9']

# for col in infection_cols:
#     table14[col] = table14[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

# binary OTHERINFECTIONSITE 
table14['OTHERINFECTIONSITE_flag'] = (
    table14['OTHERINFECTIONSITE'].fillna('').str.strip().ne('').astype(int)
)

infects_summary = table14.groupby('ACCOUNTNO').agg({
    **{col: 'max' for col in infection_cols}, 
    'OTHERINFECTIONSITE_flag': 'max'
}).reset_index()


abx14 = abx14.merge(infects_summary, on='ACCOUNTNO', how='left').fillna(0)

In [26]:
abx14.columns

Index(['ACCOUNTNO', 'Acyclovir', 'Amikacin', 'Amoxicillin',
       'Amoxicillin/Clavulanic acid', 'Amphotericin B',
       'Amphotericin B/liposome', 'Ampicillin', 'Ampicillin/Sulbactam',
       'Anidulafungin', 'Azithromycin', 'Baloxavir marboxil', 'Cefadroxil',
       'Cefazolin', 'Cefepime', 'Cefixime', 'Cefoperazone/sulbactam',
       'Cefotaxime', 'Cefoxitin', 'Ceftazidime', 'Ceftazidime/Avibactam',
       'Ceftizoxime', 'Ceftriaxone', 'Cefuroxime', 'Cephalexin',
       'Ciprofloxacin', 'Clarithromycin', 'Clindamycin', 'Colistin',
       'Dicloxacillin', 'Doripenem', 'Doxycycline', 'Ertapenem',
       'Erythromycin', 'Ethambutol', 'Famciclovir', 'Fenticonazole',
       'Fidaxomicin', 'Flomoxef', 'Fluconazole', 'Flucytosine', 'Fosfomycin',
       'Ganciclovir', 'Gentamicin', 'Griseofulvin', 'Imipenem/Cilastatin',
       'Isoniazid', 'Itraconazole', 'Lamivudine/Dolutegravir', 'Levofloxacin',
       'Linezolid', 'Meropenem', 'Metronidazole', 'Micafungin', 'Minocycline',
       'Moxif

In [27]:
abx14

,ACCOUNTNO,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Amphotericin B,Amphotericin B/liposome,Ampicillin,Ampicillin/Sulbactam,Anidulafungin,...,tenofovir/emtricitabine/bictegravir,tenofovir/emtricitabine/rilpivirine,AUTIBIOTIC_GROUP,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9,OTHERINFECTIONSITE_flag
0,I11300000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,1,0,0,0,0,0,0
1,I11300000003,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,0,0,0,0,0,0,0
2,I11300000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,0,0,0,0,0,0,0
3,I11300000011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,0,0,0,0,0,0,0
4,I11300000014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27963,I11400060686,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,1,0,0,0,0,0,0
27964,I11400060687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,0,1,0,0,0,0,0
27965,I11400060701,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,0,1,0,0,0,0,0
27966,I11400060720,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2,1,0,0,0,0,0,0


In [28]:
table14_clear = abx14.reset_index(drop=True)

In [29]:
table14_clear.to_csv('table14_clear.csv', index=False)

In [30]:
abx14 = abx14.drop(columns=['ACCOUNTNO', 'AUTIBIOTIC_GROUP'])

result = {}
for i in abx14.columns:
    summ = abx14[i].sum()
    if summ >= 100:
       result[i] = summ

for key, value in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(f'{key}: {value}')
    
print(len(result))

Amoxicillin/Clavulanic acid: 6120.0
INFECTIONSITE1: 5879
Flomoxef: 5609.0
INFECTIONSITE2: 4505
Cefixime: 3792.0
INFECTIONSITE3: 3066
Ciprofloxacin: 2725.0
Cefazolin: 2720.0
Cefuroxime: 2599.0
Piperacillin/Tazobactam: 2562.0
Azithromycin: 2531.0
Metronidazole: 1536.0
Cefoperazone/sulbactam: 1532.0
INFECTIONSITE5: 1480
Levofloxacin: 1447.0
Baloxavir marboxil: 1167.0
Peramivir: 1153.0
Cefadroxil: 1121.0
Clindamycin: 1011.0
Oseltamivir: 860.0
Ceftriaxone: 844.0
Gentamicin: 822.0
Cephalexin: 637.0
INFECTIONSITE9: 598
Doxycycline: 534.0
OTHERINFECTIONSITE_flag: 495
Ceftazidime: 474.0
Amoxicillin: 472.0
Fosfomycin: 472.0
Ampicillin: 457.0
Tenofovir alafenamide: 358.0
Fluconazole: 329.0
Vancomycin: 281.0
INFECTIONSITE4: 270
Nemonoxacin: 255.0
Cefepime: 251.0
Sulfamethoxazole/Trimethoprim: 218.0
Cefotaxime: 213.0
Nystatin: 199.0
Acyclovir: 195.0
Meropenem: 190.0
Valaciclovir: 150.0
Penicillin: 138.0
Minocycline: 119.0
Ceftizoxime: 108.0
45


In [31]:
col_sum = abx14.sum()

abx14_filter = abx14.loc[:, col_sum > 100]

In [32]:
abx14_filter.shape, abx14_filter.sum()

((27968, 45),
 Acyclovir                         195.0
 Amoxicillin                       472.0
 Amoxicillin/Clavulanic acid      6120.0
 Ampicillin                        457.0
 Azithromycin                     2531.0
 Baloxavir marboxil               1167.0
 Cefadroxil                       1121.0
 Cefazolin                        2720.0
 Cefepime                          251.0
 Cefixime                         3792.0
 Cefoperazone/sulbactam           1532.0
 Cefotaxime                        213.0
 Ceftazidime                       474.0
 Ceftizoxime                       108.0
 Ceftriaxone                       844.0
 Cefuroxime                       2599.0
 Cephalexin                        637.0
 Ciprofloxacin                    2725.0
 Clindamycin                      1011.0
 Doxycycline                       534.0
 Flomoxef                         5609.0
 Fluconazole                       329.0
 Fosfomycin                        472.0
 Gentamicin                        822.0
 L

In [33]:
mask = abx14_filter.sum(axis=1) > 0
abx14_final = abx14_filter[mask].reset_index()

In [34]:
abx14_final

,index,Acyclovir,Amoxicillin,Amoxicillin/Clavulanic acid,Ampicillin,Azithromycin,Baloxavir marboxil,Cefadroxil,Cefazolin,Cefepime,...,Tenofovir alafenamide,Valaciclovir,Vancomycin,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9,OTHERINFECTIONSITE_flag
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1,0,0,0,0,0,0
1,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27767,27963,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
27768,27964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,0,0,0,0,0
27769,27965,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,0,0,0,0,0
27770,27966,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
